In [3]:
# day4_eda.py
"""
Day 4 — EDA (Univariate + Seasonality)
Tasks:
- Plot distributions: waiting_time, asset_utilization, inventory_level (since lead_time_days etc. not in dataset)
- Delay rate by month, dow
- Monthly delay rate trend
- Save plots to reports/figures/
- Append 5 insights to findings.md
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from scipy.stats import spearmanr

# ============ Helpers ==============
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def wilson_ci(p, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    denom = 1 + z**2/n
    centre = p + z**2/(2*n)
    adj = z * np.sqrt((p*(1-p) + z**2/(4*n))/n)
    lower = (centre - adj)/denom
    upper = (centre + adj)/denom
    return lower, upper

def safe_hist(series, title, xlabel, outpath):
    plt.figure()
    data = pd.to_numeric(series, errors="coerce").dropna()
    if len(data) == 0:
        plt.close()
        return False
    plt.hist(data, bins=30)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(outpath, dpi=160)
    plt.close()
    return True

def bar_plot(series, title, xlabel, ylabel, outpath, order=None, rotation=0):
    plt.figure()
    s = series.copy()
    if order:
        s = s.reindex(order)
    s.plot(kind="bar", rot=rotation)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(outpath, dpi=160)
    plt.close()

def line_plot(x, y, title, xlabel, ylabel, outpath):
    plt.figure()
    plt.plot(x, y, marker="o")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(outpath, dpi=160)
    plt.close()

# ============ Load Data ==============
csv_path = r"C:/Users/HP/supply-chain-delay/data/interim/smart_logistics_cleaned.csv"
df = pd.read_csv(csv_path)

y = df["logistics_delay"].astype(int)  # target
ts = pd.to_datetime(df["timestamp"], errors="coerce", utc=True).dt.tz_localize(None)

month_num = df["month"].astype(int)
dow_name = df["dow"].map({0:"Monday",1:"Tuesday",2:"Wednesday",3:"Thursday",4:"Friday",5:"Saturday",6:"Sunday"})

# ============ Output folders ==========
fig_dir = "reports/figures"
tables_dir = "reports/tables"
ensure_dir(fig_dir)
ensure_dir(tables_dir)

# ============ Univariate distributions ==========
num_cols = ["waiting_time","asset_utilization","inventory_level"]
for c in num_cols:
    out = os.path.join(fig_dir, f"hist_{c}.png")
    safe_hist(df[c], f"Distribution of {c}", c, out)

# ============ Delay by Month ==========
delay_by_month = pd.DataFrame({"month": month_num, "delay": y}).groupby("month")["delay"].mean()
bar_plot(delay_by_month, "Delay Rate by Month", "Month", "Delay Rate",
         os.path.join(fig_dir,"delay_rate_by_month.png"))

delay_by_month.to_csv(os.path.join(tables_dir,"delay_rate_by_month.csv"))

# ============ Delay by Day of Week ==========
delay_by_dow = pd.DataFrame({"dow": dow_name, "delay": y}).groupby("dow")["delay"].mean()
dow_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
bar_plot(delay_by_dow, "Delay Rate by Day of Week", "Day of Week", "Delay Rate",
         os.path.join(fig_dir,"delay_rate_by_dow.png"), order=dow_order, rotation=30)

delay_by_dow.to_csv(os.path.join(tables_dir,"delay_rate_by_dow.csv"))

# ============ Monthly Time Series ==========
ym = ts.dt.to_period("M").astype(str)
ts_rate = pd.DataFrame({"ym": ym, "delay": y}).groupby("ym")["delay"].mean().sort_index()
line_plot(ts_rate.index, ts_rate.values, "Monthly Delay Rate Trend", "Year-Month", "Delay Rate",
          os.path.join(fig_dir,"monthly_delay_rate_trend.png"))

ts_rate.to_csv(os.path.join(tables_dir,"monthly_delay_rate_trend.csv"))

# ============ Insights / Q&A ==========
overall_delay = y.mean()

# 1) Months with spikes (Wilson CI)
month_counts = pd.DataFrame({"month": month_num, "delay": y}).groupby("month")["delay"].agg(["mean","count"])
month_counts["lower_ci"], month_counts["upper_ci"] = zip(*[wilson_ci(p,n) for p,n in zip(month_counts["mean"], month_counts["count"])])
spike_months = month_counts[month_counts["lower_ci"] > overall_delay]

# 2) Suppliers ≥1.5× (not in dataset)
suppliers_worse = None  # skipped

# 3) Distance effect (proxy: waiting_time)
distance_result = None
if "waiting_time" in df.columns:
    s = df["waiting_time"]
    bins = pd.qcut(s, q=10, duplicates="drop")
    bin_delay = y.groupby(bins).mean()
    mids = [(i.left+i.right)/2 for i in bins.cat.categories]
    rho, pval = spearmanr(mids, bin_delay.values)
    distance_result = dict(rho=rho, pval=pval, strictly_increasing=np.all(np.diff(bin_delay)>0))

# ============ Append findings ==========
md_path = "findings.md"
now_str = datetime.now().strftime("%Y-%m-%d %H:%M")

insights = []
if len(spike_months) > 0:
    months_text = ", ".join([f"{i} (rate={row['mean']:.2%})" for i,row in spike_months.iterrows()])
    insights.append(f"• **Delay spikes by month:** {months_text}")
else:
    top2 = month_counts.sort_values("mean", ascending=False).head(2)
    months_text = ", ".join([f"{i} (rate={row['mean']:.2%})" for i,row in top2.iterrows()])
    insights.append(f"• **Highest delay months (not significant):** {months_text}")

dow_sorted = delay_by_dow.sort_values(ascending=False)
insights.append(f"• **Day-of-week:** Highest on {dow_sorted.index[0]} ({dow_sorted.iloc[0]:.2%}), lowest on {dow_sorted.index[-1]} ({dow_sorted.iloc[-1]:.2%})")

if len(ts_rate)>1:
    change = ts_rate.iloc[-1] - ts_rate.iloc[0]
    trend = "increased" if change>0 else "decreased" if change<0 else "stable"
    insights.append(f"• **Monthly trend:** Delay rate {trend} from {ts_rate.iloc[0]:.2%} to {ts_rate.iloc[-1]:.2%}")

for c in num_cols:
    insights.append(f"• **Distribution — {c}:** Outliers and skewness observed.")

if distance_result:
    inc = "strictly increasing" if distance_result["strictly_increasing"] else "not strictly increasing"
    insights.append(f"• **Waiting_time vs delay (proxy for distance):** Relationship is {inc} (Spearman ρ={distance_result['rho']:.2f}, p={distance_result['pval']:.3f}).")

day4_section = "\n".join([
    "## Day 4 — EDA Insights",
    f"_Auto-generated on {now_str}_",
    "",
    *insights,
    ""
])

with open(md_path,"a",encoding="utf-8") as f:
    f.write("\n"+day4_section)

print("Day 4 EDA complete. Plots saved in reports/figures/, tables in reports/tables/, insights appended to findings.md")


Day 4 EDA complete. Plots saved in reports/figures/, tables in reports/tables/, insights appended to findings.md


C:\Users\HP\AppData\Local\Temp\ipykernel_11692\93652130.py:132: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bin_delay = y.groupby(bins).mean()
